# CustomerIQ: RFM Analysis Deep Dive

This notebook implements comprehensive RFM (Recency, Frequency, Monetary) analysis for customer segmentation in the CustomerIQ platform.

## Objectives:
- Calculate RFM scores for all customers
- Implement dynamic customer segmentation
- Analyze segment characteristics and behaviors
- Generate actionable business recommendations
- Visualize customer segments and their performance

## 1. Setup and Data Loading

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# System imports
import sys
from pathlib import Path

# Add project path
project_root = Path.cwd().parent
src_path = project_root / 'src'
sys.path.append(str(src_path))

# Import custom modules
from rfm_analyzer import RFMAnalyzer
from utils.visualization import CustomerVisualizer

print("✅ Libraries loaded successfully!")

In [ ]:
# Load processed data from previous notebook
data_dir = project_root / 'data' / 'processed'

try:
    customer_data = pd.read_csv(data_dir / 'customer_features.csv', index_col=0)
    transaction_data = pd.read_csv(data_dir / 'clean_transactions.csv')
    print(f"✅ Data loaded successfully!")
    print(f"Customer data shape: {customer_data.shape}")
    print(f"Transaction data shape: {transaction_data.shape}")
except FileNotFoundError:
    print("❌ Processed data not found. Please run the data exploration notebook first.")
    # Create sample data as fallback
    from data_processor import DataProcessor
    data_processor = DataProcessor()
    transaction_data = data_processor.create_sample_dataset(n_customers=1000)
    customer_data = data_processor.create_customer_features(transaction_data)
    print("✅ Created sample data for analysis")

# Initialize RFM analyzer
rfm_analyzer = RFMAnalyzer(quantiles=5)
visualizer = CustomerVisualizer()

## 2. RFM Score Calculation

In [ ]:
# Calculate RFM scores
print("📊 CALCULATING RFM SCORES")
print("=" * 50)

# Calculate RFM
rfm_data = rfm_analyzer.calculate_rfm(customer_data)

print(f"RFM analysis completed for {len(rfm_data)} customers")
print("\nRFM Components Summary:")
print(f"Recency - Min: {rfm_data['Recency'].min():.0f} days, Max: {rfm_data['Recency'].max():.0f} days")
print(f"Frequency - Min: {rfm_data['Frequency'].min():.0f} orders, Max: {rfm_data['Frequency'].max():.0f} orders")
print(f"Monetary - Min: ${rfm_data['Monetary'].min():.2f}, Max: ${rfm_data['Monetary'].max():.2f}")

# Display RFM score distribution
print("\nRFM Score Distribution:")
for score in ['R_Score', 'F_Score', 'M_Score']:
    distribution = rfm_data[score].value_counts().sort_index()
    print(f"\n{score}:")
    for value, count in distribution.items():
        percentage = (count / len(rfm_data)) * 100
        print(f"  Score {value}: {count:,} customers ({percentage:.1f}%)")

rfm_data.head()

In [ ]:
# Visualize RFM distributions
fig = visualizer.plot_rfm_distribution(rfm_data)
fig.show()

# Additional RFM analysis plots
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('RFM Score Correlation', 'Recency vs Frequency', 
                   'Frequency vs Monetary', 'Recency vs Monetary')
)

# RFM Score correlation heatmap
rfm_corr = rfm_data[['R_Score', 'F_Score', 'M_Score']].corr()
fig.add_trace(
    go.Heatmap(
        z=rfm_corr.values,
        x=rfm_corr.columns,
        y=rfm_corr.columns,
        colorscale='RdBu',
        text=np.around(rfm_corr.values, decimals=2),
        texttemplate="%{text}",
        showscale=False
    ),
    row=1, col=1
)

# Scatter plots
fig.add_trace(
    go.Scatter(
        x=rfm_data['Recency'], 
        y=rfm_data['Frequency'],
        mode='markers',
        marker=dict(size=4, opacity=0.6),
        name='Customers'
    ),
    row=1, col=2
)

fig.add_trace(
    go.Scatter(
        x=rfm_data['Frequency'], 
        y=rfm_data['Monetary'],
        mode='markers',
        marker=dict(size=4, opacity=0.6),
        name='Customers'
    ),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(
        x=rfm_data['Recency'], 
        y=rfm_data['Monetary'],
        mode='markers',
        marker=dict(size=4, opacity=0.6),
        name='Customers'
    ),
    row=2, col=2
)

fig.update_layout(height=600, showlegend=False, title_text="RFM Component Analysis")
fig.show()

## 3. Customer Segmentation

In [ ]:
# Perform customer segmentation
print("👥 CUSTOMER SEGMENTATION")
print("=" * 50)

segmented_data, segment_stats = rfm_analyzer.segment_customers(rfm_data)

print("Customer segments created successfully!")
print(f"\nSegment Distribution:")
segment_counts = segmented_data['Segment'].value_counts()
for segment, count in segment_counts.items():
    percentage = (count / len(segmented_data)) * 100
    print(f"  {segment}: {count:,} customers ({percentage:.1f}%)")

# Display segment statistics
print("\n📊 Segment Performance Statistics:")
display(segment_stats)

segmented_data.head()

In [ ]:
# Visualize customer segments
fig = visualizer.plot_customer_segments(segmented_data)
fig.show()

# Segment comparison charts
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Revenue by Segment', 'Customer Count by Segment', 
                   'Average RFM Scores by Segment', 'Customer Percentage by Segment')
)

# Revenue by segment
fig.add_trace(
    go.Bar(
        x=segment_stats['segment'],
        y=segment_stats['total_monetary'],
        name='Total Revenue',
        marker_color='lightblue'
    ),
    row=1, col=1
)

# Customer count by segment
fig.add_trace(
    go.Bar(
        x=segment_stats['segment'],
        y=segment_stats['customer_count'],
        name='Customer Count',
        marker_color='lightgreen'
    ),
    row=1, col=2
)

# Average RFM scores by segment
fig.add_trace(
    go.Bar(
        x=segment_stats['segment'],
        y=segment_stats['avg_rfm_score'],
        name='Avg RFM Score',
        marker_color='coral'
    ),
    row=2, col=1
)

# Customer percentage pie chart
fig.add_trace(
    go.Pie(
        labels=segment_stats['segment'],
        values=segment_stats['customer_percentage'],
        name='Customer %'
    ),
    row=2, col=2
)

fig.update_layout(height=700, title_text="Customer Segment Analysis")
fig.show()

## 4. Segment Deep Dive Analysis

In [ ]:
# Detailed analysis of each segment
print("🔍 SEGMENT DEEP DIVE ANALYSIS")
print("=" * 50)

for segment in segmented_data['Segment'].unique():
    segment_customers = segmented_data[segmented_data['Segment'] == segment]
    
    print(f"\n🎯 {segment.upper()}")
    print("-" * 40)
    print(f"Customer Count: {len(segment_customers):,}")
    print(f"Percentage of Total: {len(segment_customers)/len(segmented_data)*100:.1f}%")
    
    # RFM characteristics
    print(f"\nRFM Characteristics:")
    print(f"  Average Recency: {segment_customers['Recency'].mean():.1f} days")
    print(f"  Average Frequency: {segment_customers['Frequency'].mean():.1f} orders")
    print(f"  Average Monetary: ${segment_customers['Monetary'].mean():.2f}")
    
    # Business metrics
    total_revenue = segment_customers['Monetary'].sum()
    avg_order_value = segment_customers['Monetary'].sum() / segment_customers['Frequency'].sum()
    
    print(f"\nBusiness Impact:")
    print(f"  Total Revenue: ${total_revenue:,.2f}")
    print(f"  Revenue Share: {total_revenue/segmented_data['Monetary'].sum()*100:.1f}%")
    print(f"  Average Order Value: ${avg_order_value:.2f}")
    
    # RFM score distribution within segment
    print(f"\nRFM Score Distribution:")
    print(f"  R Score - Mode: {segment_customers['R_Score'].mode().iloc[0] if not segment_customers['R_Score'].mode().empty else 'N/A'}")
    print(f"  F Score - Mode: {segment_customers['F_Score'].mode().iloc[0] if not segment_customers['F_Score'].mode().empty else 'N/A'}")
    print(f"  M Score - Mode: {segment_customers['M_Score'].mode().iloc[0] if not segment_customers['M_Score'].mode().empty else 'N/A'}")

In [ ]:
# Create segment comparison heatmap
segment_comparison = segmented_data.groupby('Segment').agg({
    'Recency': 'mean',
    'Frequency': 'mean', 
    'Monetary': 'mean',
    'R_Score': 'mean',
    'F_Score': 'mean',
    'M_Score': 'mean',
    'RFM_Score_Numeric': 'mean'
}).round(2)

# Normalize for better visualization
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
segment_comparison_normalized = pd.DataFrame(
    scaler.fit_transform(segment_comparison),
    index=segment_comparison.index,
    columns=segment_comparison.columns
)

# Create heatmap
fig = go.Figure(data=go.Heatmap(
    z=segment_comparison_normalized.values,
    x=segment_comparison_normalized.columns,
    y=segment_comparison_normalized.index,
    colorscale='RdYlBu_r',
    text=segment_comparison.values,
    texttemplate="%{text}",
    textfont={"size": 10},
    showscale=True
))

fig.update_layout(
    title="Customer Segment Characteristics Heatmap (Normalized)",
    xaxis_title="Metrics",
    yaxis_title="Customer Segments",
    height=500
)

fig.show()

print("\n📊 Raw Segment Comparison:")
display(segment_comparison)

## 5. Business Recommendations and Action Plans

In [ ]:
# Generate recommendations for each segment
print("💡 BUSINESS RECOMMENDATIONS")
print("=" * 50)

segmented_with_actions, recommendations = rfm_analyzer.recommend_actions(segmented_data)

for segment, recommendation in recommendations.items():
    segment_count = len(segmented_data[segmented_data['Segment'] == segment])
    if segment_count > 0:
        print(f"\n🎯 {segment.upper()}")
        print(f"📊 Customer Count: {segment_count:,}")
        print(f"🎪 Strategy: {recommendation['strategy']}")
        print(f"⚡ Priority: {recommendation['priority']}")
        print(f"📋 Recommended Actions:")
        for i, action in enumerate(recommendation['actions'], 1):
            print(f"   {i}. {action}")
        print("-" * 60)

In [ ]:
# Calculate potential business impact
print("💰 POTENTIAL BUSINESS IMPACT ANALYSIS")
print("=" * 50)

# High-priority segments analysis
high_priority_segments = ['Champions', 'Cannot Lose Them', 'At Risk', 'Loyal Customers']
high_priority_customers = segmented_data[segmented_data['Segment'].isin(high_priority_segments)]

print(f"High Priority Customer Analysis:")
print(f"  Total High Priority Customers: {len(high_priority_customers):,}")
print(f"  Percentage of Customer Base: {len(high_priority_customers)/len(segmented_data)*100:.1f}%")
print(f"  Total Revenue from High Priority: ${high_priority_customers['Monetary'].sum():,.2f}")
print(f"  Revenue Share: {high_priority_customers['Monetary'].sum()/segmented_data['Monetary'].sum()*100:.1f}%")

# At Risk customer analysis
at_risk_customers = segmented_data[segmented_data['Segment'] == 'At Risk']
cannot_lose_customers = segmented_data[segmented_data['Segment'] == 'Cannot Lose Them']

revenue_at_risk = at_risk_customers['Monetary'].sum() + cannot_lose_customers['Monetary'].sum()
customers_at_risk = len(at_risk_customers) + len(cannot_lose_customers)

print(f"\nRetention Opportunity:")
print(f"  Customers at Risk: {customers_at_risk:,}")
print(f"  Revenue at Risk: ${revenue_at_risk:,.2f}")
print(f"  Average Revenue per At-Risk Customer: ${revenue_at_risk/customers_at_risk:.2f}")

# Growth opportunity analysis
new_customers = segmented_data[segmented_data['Segment'] == 'New Customers']
potential_loyalists = segmented_data[segmented_data['Segment'] == 'Potential Loyalists']

growth_customers = len(new_customers) + len(potential_loyalists)
current_growth_revenue = new_customers['Monetary'].sum() + potential_loyalists['Monetary'].sum()

print(f"\nGrowth Opportunity:")
print(f"  Growth Potential Customers: {growth_customers:,}")
print(f"  Current Revenue from Growth Segment: ${current_growth_revenue:,.2f}")
print(f"  Potential Revenue if converted to Loyal: ${current_growth_revenue * 2:.2f} (estimated 2x increase)")

## 6. Advanced RFM Analysis

In [ ]:
# Identify high-value customers
print("💎 HIGH-VALUE CUSTOMER IDENTIFICATION")
print("=" * 50)

high_value_customers = rfm_analyzer.identify_high_value_customers(segmented_data, top_percentage=0.1)

print(f"Top 10% High-Value Customers:")
print(f"  Customer Count: {len(high_value_customers):,}")
print(f"  Total Revenue: ${high_value_customers['Monetary'].sum():,.2f}")
print(f"  Average Revenue per Customer: ${high_value_customers['Monetary'].mean():.2f}")
print(f"  Revenue Share: {high_value_customers['Monetary'].sum()/segmented_data['Monetary'].sum()*100:.1f}%")

# Segment distribution of high-value customers
high_value_segments = high_value_customers['Segment'].value_counts()
print(f"\nHigh-Value Customer Segment Distribution:")
for segment, count in high_value_segments.items():
    percentage = (count / len(high_value_customers)) * 100
    print(f"  {segment}: {count:,} ({percentage:.1f}%)")

display(high_value_customers[['Recency', 'Frequency', 'Monetary', 'RFM_Score_Numeric', 'Segment']].head(10))

In [ ]:
# RFM Score distribution analysis
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('RFM Score Distribution', 'Top RFM Scores', 
                   'RFM Score vs Revenue', 'Segment vs RFM Score')
)

# Overall RFM score distribution
fig.add_trace(
    go.Histogram(x=segmented_data['RFM_Score_Numeric'], nbinsx=30, name='RFM Distribution'),
    row=1, col=1
)

# Top RFM scores (top 20)
top_rfm = segmented_data.nlargest(20, 'RFM_Score_Numeric')
fig.add_trace(
    go.Bar(x=range(1, 21), y=top_rfm['RFM_Score_Numeric'], name='Top 20 RFM Scores'),
    row=1, col=2
)

# RFM Score vs Revenue scatter
fig.add_trace(
    go.Scatter(
        x=segmented_data['RFM_Score_Numeric'], 
        y=segmented_data['Monetary'],
        mode='markers',
        marker=dict(size=4, opacity=0.6),
        name='RFM vs Revenue'
    ),
    row=2, col=1
)

# Box plot: Segment vs RFM Score
for segment in segmented_data['Segment'].unique():
    segment_data = segmented_data[segmented_data['Segment'] == segment]
    fig.add_trace(
        go.Box(y=segment_data['RFM_Score_Numeric'], name=segment),
        row=2, col=2
    )

fig.update_layout(height=700, title_text="Advanced RFM Analysis")
fig.show()

## 7. Segment Migration Analysis (Simulation)

In [ ]:
# Simulate segment migration over time
print("📈 SEGMENT MIGRATION SIMULATION")
print("=" * 50)

# Create a simulated "previous period" by adding noise to current data
np.random.seed(42)
previous_period_data = segmented_data.copy()

# Add some realistic changes to simulate time passage
# Increase recency for some customers (they haven't purchased recently)
recency_increase = np.random.exponential(10, len(previous_period_data))
previous_period_data['Recency'] = previous_period_data['Recency'] + recency_increase

# Slightly decrease frequency for some customers
frequency_decrease = np.random.poisson(0.5, len(previous_period_data))
previous_period_data['Frequency'] = np.maximum(1, previous_period_data['Frequency'] - frequency_decrease)

# Recalculate RFM scores for previous period
previous_rfm = rfm_analyzer.calculate_rfm(previous_period_data)
previous_segmented, _ = rfm_analyzer.segment_customers(previous_rfm)

# Calculate migration matrix
migration_analysis = rfm_analyzer.calculate_segment_migration(
    segmented_data, previous_segmented
)

print("Segment Migration Matrix (from previous to current period):")
display(migration_analysis['migration_percentages'].round(1))

print(f"\nCustomers who changed segments: {len(migration_analysis['segment_changes'])}")
print("\nTop Segment Changes:")
change_summary = migration_analysis['segment_changes'].groupby(
    ['Segment_previous', 'Segment_current']
).size().reset_index(name='count').sort_values('count', ascending=False)

display(change_summary.head(10))

## 8. Export and Summary

In [ ]:
# Create comprehensive RFM summary
print("📋 RFM ANALYSIS SUMMARY")
print("=" * 60)

summary_stats = {
    'Total Customers Analyzed': len(segmented_data),
    'Number of Segments': segmented_data['Segment'].nunique(),
    'Total Revenue': f"${segmented_data['Monetary'].sum():,.2f}",
    'Average Customer Value': f"${segmented_data['Monetary'].mean():.2f}",
    'Average RFM Score': f"{segmented_data['RFM_Score_Numeric'].mean():.2f}",
    'High-Value Customers (Top 10%)': len(high_value_customers),
    'Revenue from Top 10%': f"${high_value_customers['Monetary'].sum():,.2f}",
    'Revenue Concentration (Top 10%)': f"{high_value_customers['Monetary'].sum()/segmented_data['Monetary'].sum()*100:.1f}%"
}

for metric, value in summary_stats.items():
    print(f"{metric:.<40} {value}")

print("\n🎯 KEY INSIGHTS:")
print("-" * 60)

# Generate key insights
largest_segment = segment_stats.loc[segment_stats['customer_count'].idxmax(), 'segment']
highest_revenue_segment = segment_stats.loc[segment_stats['total_monetary'].idxmax(), 'segment']
best_rfm_segment = segment_stats.loc[segment_stats['avg_rfm_score'].idxmax(), 'segment']

print(f"• Largest segment: {largest_segment} ({segment_stats.loc[segment_stats['segment']==largest_segment, 'customer_percentage'].iloc[0]:.1f}% of customers)")
print(f"• Highest revenue segment: {highest_revenue_segment} (${segment_stats.loc[segment_stats['segment']==highest_revenue_segment, 'total_monetary'].iloc[0]:,.0f})")
print(f"• Best RFM segment: {best_rfm_segment} (score: {segment_stats.loc[segment_stats['segment']==best_rfm_segment, 'avg_rfm_score'].iloc[0]:.2f})")
print(f"• Revenue concentration: Top 10% customers generate {high_value_customers['Monetary'].sum()/segmented_data['Monetary'].sum()*100:.1f}% of revenue")
print(f"• Customers at risk: {customers_at_risk:,} customers with ${revenue_at_risk:,.0f} revenue at stake")

# Save RFM analysis results
results_dir = project_root / 'data' / 'processed'
results_dir.mkdir(parents=True, exist_ok=True)

# Export files
segmented_data.to_csv(results_dir / 'rfm_segmented_customers.csv', index=True)
segment_stats.to_csv(results_dir / 'rfm_segment_statistics.csv', index=False)
high_value_customers.to_csv(results_dir / 'high_value_customers.csv', index=True)

print("\n💾 Files saved:")
print(f"• RFM segmented customers: rfm_segmented_customers.csv")
print(f"• Segment statistics: rfm_segment_statistics.csv")
print(f"• High-value customers: high_value_customers.csv")

print("\n✅ RFM Analysis completed successfully!")
print("🔄 Next step: Proceed to Churn Prediction notebook for risk analysis.")